# 任务4优化版：原尺寸 ResNet34 U-Net，完整五折，EMA与D4对照

**只需上传这一个Notebook，并挂载任务4比赛数据。** 启用GPU；首次下载torchvision官方ImageNet ResNet34权重需打开Internet，也可挂载官方`resnet34-b627a593.pth`并填写`WEIGHTS_PATH`。无需安装SMP、timm或albumentations。本Notebook内嵌全部项目实现。

### 本轮优化

1. **通用ImageNet编码器＋保留全分辨率的U-Net解码器。** 随机初始化分割头，未使用血管分割专用权重或外部眼底标签。RGB按ImageNet均值/方差归一化，BatchNorm运行统计冻结，编码器使用较小学习率。
2. **原始565×584像素上采样patch。** 不先压成512训练；针对细血管增加全分辨率浅层跳接与辅助监督。训练增强与标签同步；测试先得到原尺寸概率，再缩放概率至512、最后二值化。
3. **完整五折OOF＋五模型等权测试平均。** 每张训练图仅在未训练过它的那一折产生OOF预测；与首版固定同一划分。80轮和最终EMA预先固定，监控分数不用于选轮数，不早停。
4. **同一权重比较无TTA与D4。** TTA包含旋转/翻转共8种，逐patch准确反变换。仅当固定0.5阈值下OOF平均Dice至少提高0.001才选D4，否则主提交使用无TTA。阈值仅在OOF上调节。

这是待运行的优化候选，尚无完整正式五折成绩，不能保证高于首版。原首版的0.784356只来自4张验证图；不能直接与新20图OOF分数作同范围比较。结果表另列相同4图的固定0.5阈值对照。

### 标签与评分细节

原始JPEG标签是白血管、黑背景，截图却写血管0；官方转换脚本编码非零像素。主CSV沿用实际标签/官方脚本的白血管前景约定，另提供原尺寸黑血管PNG。训练目标为原图灰度>127，避免JPEG压缩噪声扩张血管。

主指标为512尺寸干净血管目标的逐图平均Dice，同时记录原尺寸Dice、原JPEG经官方双线性缩放及>0所得目标的代理Dice。`submission_converter_proxy.csv`只针对后者选阈值，**不是已确认更符合私榜的提交**。私榜真值预处理仍未知。

TTA方式与全局阈值由OOF选择，所选分数存在选择偏差。跨折阈值诊断并非完整嵌套独立测试；表中同时保留固定0.5的OOF分数。五折平均的测试性能仍需真实比赛反馈。

来源：[torchvision ResNet34官方权重与预处理](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet34.html)、[PyTorch混合精度](https://docs.pytorch.org/docs/stable/notes/amp_examples.html)。未提供完整任务4 Rules，是否允许通用预训练以比赛规则为准。


## 1. 运行配置

默认单GPU依次训练5折，每折80轮，每轮256个patch。实际耗时取决于GPU，第一轮会给出实测估计，D4预测与打包另需时间。无需开启多GPU分布式，也不会同时占用两块GPU。

先跑默认配置；如只检查云端环境，可设`SMOKE_MODE=True`，不会生成完整五折主提交。配置或源码改变时必须换输出目录；不能把不同设置的折拼成同一OOF。


In [ ]:
from pathlib import Path

# 数据根目录应直接包含train/label、train/image、test/image、sample_submission.csv。
DATA_DIR = None                         # 自动查找；多个副本时填写实际segmentation目录
OUTPUT_DIR = Path('/kaggle/working/task4_v2_results') if Path('/kaggle/working').exists() else Path.cwd() / 'task4_v2_results'
WEIGHTS_PATH = None                     # 无网络时指向官方resnet34-b627a593.pth
RESUME_FROM = None                      # 可填已挂载的完整结果目录或full_backup.zip

SEED = 42
EPOCHS = 80                             # 固定最终轮次；不按外层验证选最佳轮次
PATCH_SIZE = 256
PATCHES_PER_EPOCH = 256
BATCH_SIZE = 8                          # 显存不足可改4；改配置须换OUTPUT_DIR
FOLDS_TO_RUN = [0, 1, 2, 3, 4]          # 中断后保留原列表即可，已完成折自动复用
PRETRAINED = True                       # 通用ImageNet编码器，不是血管分割专用权重
RUN_TRAINING = True
SMOKE_MODE = False                      # True只跑第0折、1轮，输出单独_smoke目录
SAVE_FULL_BACKUP = True                 # 默认完整备份包含权重、优化器和官方初始权重


## 2. 加载内嵌实现（无需上传其他Python文件）


In [ ]:
import importlib.util
import sys

RUNTIME_DIR = (Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()) / 'task4_v2_runtime'
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
BASE_SOURCE = '"""Task 4: small-data retinal vessel segmentation, audit through submission.\n\nOnly the supplied training images/labels are used. No downloaded weights.\nThe original segmentation_to_csv.py is kept unchanged and checked on export.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageDraw\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\n\n@dataclass\nclass Config:\n    seed: int = 42\n    epochs: int = 80\n    size: int = 512\n    patch: int = 256\n    patches_per_epoch: int = 128\n    batch_size: int = 8\n    base_channels: int = 24\n    lr: float = 0.001\n    weight_decay: float = 0.0001\n    folds: int = 5\n    fold: int = 0\n    threads: int = 4\n    amp: bool = True\n\n\ndef save_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    os.replace(tmp, path)\n\n\ndef file_sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef find_data(data_dir=None):\n    if data_dir:\n        candidates = [Path(data_dir)]\n    else:\n        base = Path(__file__).resolve().parent if \'__file__\' in globals() else Path.cwd()\n        candidates = [base / \'2026task-4/segmentation\', Path.cwd() / \'2026task-4/segmentation\',\n                      Path.cwd() / \'task4/2026task-4/segmentation\']\n        if Path(\'/kaggle/input\').exists():\n            candidates += [p.parent for p in Path(\'/kaggle/input\').rglob(\'sample_submission.csv\')]\n    found = {p.resolve() for p in candidates if (p / \'train/label\').is_dir()\n             and (p / \'test/image\').is_dir() and (p / \'sample_submission.csv\').is_file()}\n    if len(found) != 1:\n        raise ValueError(f\'Set data_dir to the segmentation folder. Found: {sorted(map(str, found))}\')\n    return found.pop()\n\n\ndef list_images(directory):\n    files = sorted(Path(directory).glob(\'*.jpg\'), key=lambda p: int(p.stem))\n    if not files:\n        raise ValueError(f\'No jpg images: {directory}\')\n    return files\n\n\ndef strict_label(path, size=512):\n    # JPEG labels contain ringing; >0 would thicken the foreground substantially.\n    mask = (np.asarray(Image.open(path).convert(\'L\')) > 127).astype(np.uint8)\n    return cv2.resize(mask, (size, size), interpolation=cv2.INTER_NEAREST).astype(bool)\n\n\ndef converter_label(path):\n    # Exact target IF hidden ground truth was processed by the supplied converter.\n    # This is a diagnostic proxy, not an assertion about the private scoring code.\n    return np.asarray(Image.open(path).convert(\'L\').resize((512, 512), Image.Resampling.BILINEAR)) > 0\n\n\ndef make_features(path, size=512):\n    rgb = np.asarray(Image.open(path).convert(\'RGB\'))\n    rgb = cv2.resize(rgb, (size, size), interpolation=cv2.INTER_AREA)\n    green = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(rgb[:, :, 1])\n    return np.concatenate([rgb, green[:, :, None]], axis=2)\n\n\ndef audit(root, output, cfg):\n    output.mkdir(parents=True, exist_ok=True)\n    train, test = list_images(root / \'train/image\'), list_images(root / \'test/image\')\n    labels = {p.stem: p for p in list_images(root / \'train/label\')}\n    if len(train) != 20 or len(test) != 20 or set(labels) != {p.stem for p in train}:\n        raise ValueError(\'Expected 20 train image/label pairs and 20 test images.\')\n    with (root / \'sample_submission.csv\').open(newline=\'\', encoding=\'utf-8-sig\') as f:\n        reader = csv.DictReader(f)\n        if reader.fieldnames != [\'Id\', \'Predicted\']:\n            raise ValueError(f\'Unexpected template columns: {reader.fieldnames}\')\n        ids = [r[\'Id\'] for r in reader]\n    if len(ids) != len(set(ids)) or set(ids) != {p.stem for p in test}:\n        raise ValueError(\'Template IDs do not match test images.\')\n    rows, groups, test_hashes = [], {}, set()\n    for split, paths in [(\'train\', train), (\'test\', test)]:\n        for p in paths:\n            im = Image.open(p).convert(\'RGB\')\n            digest = hashlib.sha256(np.asarray(im).tobytes()).hexdigest()\n            row = dict(split=split, id=p.stem, width=im.width, height=im.height,\n                       image_sha256=file_sha(p), pixel_sha256=digest)\n            if split == \'train\':\n                if Image.open(labels[p.stem]).size != im.size:\n                    raise ValueError(f\'Image/label size mismatch: {p.stem}\')\n                row.update(label_sha256=file_sha(labels[p.stem]),\n                           vessel_fraction=float(strict_label(labels[p.stem]).mean()),\n                           converter_fraction=float(converter_label(labels[p.stem]).mean()))\n                groups.setdefault(digest, []).append(p.stem)\n            else:\n                test_hashes.add(digest)\n            rows.append(row)\n    if set(groups) & test_hashes:\n        raise ValueError(\'Exact decoded train/test duplicate detected; inspect before training.\')\n    ordered = sorted(groups)\n    random.Random(cfg.seed).shuffle(ordered)\n    assignments = {}\n    sizes = [0] * cfg.folds\n    for digest in ordered:\n        fold = min(range(cfg.folds), key=lambda k: sizes[k])\n        for image_id in groups[digest]:\n            assignments[image_id] = fold\n        sizes[fold] += len(groups[digest])\n    for row in rows:\n        row[\'fold\'] = assignments.get(row[\'id\'], -1) if row[\'split\'] == \'train\' else -1\n    fields = sorted({key for row in rows for key in row})\n    with (output / \'manifest.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fields)\n        writer.writeheader()\n        writer.writerows(rows)\n    manifest_fingerprint = hashlib.sha256(json.dumps(rows, sort_keys=True).encode()).hexdigest()\n    summary = dict(train_images=len(train), test_images=len(test), unique_train_groups=len(groups),\n                   folds=sizes, validation_ids=[p.stem for p in train if assignments[p.stem] == cfg.fold],\n                   decoded_train_test_overlap=0, patient_ids_available=False,\n                   data_fingerprint=manifest_fingerprint,\n                   observed_label_polarity=\'white vessels / black background\',\n                   screenshot_label_polarity=\'black vessels / white background\',\n                   converter=dict(size=[512, 512], resize=\'PIL bilinear\', foreground=\'nonzero\',\n                                  flatten=\'row-major C order\', start_index=1,\n                                  source_sha256=file_sha(root / \'segmentation_to_csv.py\')),\n                   strict_label=\'native grayscale >127, then nearest resize\',\n                   mean_clean_foreground=float(np.mean([r[\'vessel_fraction\'] for r in rows if r[\'split\']==\'train\'])),\n                   mean_converter_foreground=float(np.mean([r[\'converter_fraction\'] for r in rows if r[\'split\']==\'train\'])))\n    save_json(output / \'audit_summary.json\', summary)\n    return train, test, labels, assignments, ids, summary\n\n\ndef paired_d4(image, mask, rotation, flip):\n    image, mask = np.rot90(image, rotation), np.rot90(mask, rotation)\n    if flip:\n        image, mask = np.flip(image, 1), np.flip(mask, 1)\n    return image.copy(), mask.copy()\n\n\nclass PatchDataset(Dataset):\n    def __init__(self, paths, labels, cfg):\n        self.cfg = cfg\n        self.images = [make_features(p, cfg.size) for p in paths]\n        self.masks = [strict_label(labels[p.stem], cfg.size).astype(np.float32) for p in paths]\n        self.positive = [np.argwhere(m > 0) for m in self.masks]\n\n    def __len__(self):\n        return self.cfg.patches_per_epoch\n\n    def __getitem__(self, index):\n        idx = random.randrange(len(self.images))\n        image, mask = self.images[idx], self.masks[idx]\n        size, patch = self.cfg.size, self.cfg.patch\n        crop = random.randint(patch, min(size, int(patch * 1.25)))\n        if random.random() < 0.3 and len(self.positive[idx]):\n            y, x = self.positive[idx][random.randrange(len(self.positive[idx]))]\n            top, left = int(np.clip(y - crop//2, 0, size-crop)), int(np.clip(x-crop//2, 0, size-crop))\n        else:\n            top, left = random.randint(0, size-crop), random.randint(0, size-crop)\n        image = cv2.resize(image[top:top+crop, left:left+crop], (patch, patch), interpolation=cv2.INTER_LINEAR)\n        mask = cv2.resize(mask[top:top+crop, left:left+crop], (patch, patch), interpolation=cv2.INTER_NEAREST)\n        image, mask = paired_d4(image, mask, random.randrange(4), random.random() < 0.5)\n        image = image.astype(np.float32) / 255.0\n        image = np.clip(image * random.uniform(0.9, 1.1) + random.uniform(-0.04, 0.04), 0, 1)\n        image = image ** random.uniform(0.85, 1.15)\n        return torch.from_numpy(image.transpose(2, 0, 1).copy()) * 2 - 1, torch.from_numpy(mask[None].copy())\n\n\nclass ConvBlock(nn.Sequential):\n    def __init__(self, cin, cout):\n        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU(),\n                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU())\n\n\nclass VesselUNet(nn.Module):\n    def __init__(self, base=24):\n        super().__init__()\n        ch = [base * (2**i) for i in range(5)]\n        self.enc = nn.ModuleList([ConvBlock(4 if i == 0 else ch[i-1], ch[i]) for i in range(5)])\n        self.dec = nn.ModuleList([ConvBlock(ch[i+1]+ch[i], ch[i]) for i in range(3, -1, -1)])\n        self.head = nn.Conv2d(base, 1, 1)\n\n    def forward(self, x):\n        skips = []\n        for i, block in enumerate(self.enc):\n            x = block(x if i == 0 else F.max_pool2d(x, 2))\n            skips.append(x)\n        for block, skip in zip(self.dec, reversed(skips[:-1])):\n            x = F.interpolate(x, size=skip.shape[-2:], mode=\'bilinear\', align_corners=False)\n            x = block(torch.cat([x, skip], 1))\n        return self.head(x)\n\n\ndef segmentation_loss(logits, target):\n    bce = F.binary_cross_entropy_with_logits(logits.float(), target.float())\n    prob = logits.float().sigmoid()\n    dice = (2*(prob*target).sum((1,2,3))+1) / (prob.sum((1,2,3))+target.sum((1,2,3))+1)\n    return 0.5*bce + 0.5*(1-dice.mean())\n\n\ndef dice_score(pred, target):\n    pred, target = np.asarray(pred, bool), np.asarray(target, bool)\n    return float((2*np.logical_and(pred, target).sum()+1e-7) / (pred.sum()+target.sum()+1e-7))\n\n\n@torch.inference_mode()\ndef predict(model, path, cfg, device):\n    model.eval()\n    image = make_features(path, cfg.size)\n    step = cfg.patch // 2\n    starts = list(range(0, cfg.size-cfg.patch+1, step))\n    if starts[-1] != cfg.size-cfg.patch:\n        starts.append(cfg.size-cfg.patch)\n    coords = [(y,x) for y in starts for x in starts]\n    weight = np.maximum(np.outer(np.hanning(cfg.patch), np.hanning(cfg.patch)), 0.05).astype(np.float32)\n    sums, weights = np.zeros((cfg.size, cfg.size), np.float32), np.zeros((cfg.size, cfg.size), np.float32)\n    for start in range(0, len(coords), cfg.batch_size):\n        positions = coords[start:start+cfg.batch_size]\n        patches = np.stack([image[y:y+cfg.patch, x:x+cfg.patch].transpose(2,0,1) for y,x in positions])\n        tensor = torch.from_numpy(patches.copy()).to(device).float()/127.5-1\n        with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type==\'cuda\'):\n            probs = model(tensor).float().sigmoid()[:,0].cpu().numpy()\n        for (y,x), prob in zip(positions, probs):\n            sums[y:y+cfg.patch,x:x+cfg.patch] += prob * weight\n            weights[y:y+cfg.patch,x:x+cfg.patch] += weight\n    return sums/weights\n\n\ndef atomic_checkpoint(path, value):\n    tmp = path.with_suffix(\'.tmp\')\n    torch.save(value, tmp)\n    os.replace(tmp, path)\n\n\ndef rng_state():\n    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),\n                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)\n\n\ndef restore_rng(state):\n    random.setstate(state[\'python\'])\n    np.random.set_state(state[\'numpy\'])\n    torch.set_rng_state(state[\'torch\'].cpu())\n    if torch.cuda.is_available() and state[\'cuda\'] is not None:\n        torch.cuda.set_rng_state_all([s.cpu() for s in state[\'cuda\']])\n\n\ndef train_stage(paths, val_paths, labels, cfg, folder, device, epochs):\n    folder.mkdir(parents=True, exist_ok=True)\n    seed_all(cfg.seed)\n    dataset = PatchDataset(paths, labels, cfg)\n    loader = DataLoader(dataset, batch_size=cfg.batch_size, num_workers=0, shuffle=False,\n                        pin_memory=device.type==\'cuda\')\n    model = VesselUNet(cfg.base_channels).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)\n    # Full retraining retains the original 80-epoch cosine horizon.\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=cfg.lr*0.02)\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=cfg.amp and device.type==\'cuda\')\n    best, history, first = -1.0, [], 1\n    last = folder / \'last.pt\'\n    if last.exists():\n        ckpt = torch.load(last, map_location=\'cpu\', weights_only=False)\n        if ckpt[\'train_ids\'] != [p.stem for p in paths] or ckpt[\'target_epochs\'] != epochs:\n            raise ValueError(\'Stage mismatch. Use a new output directory.\')\n        model.load_state_dict(ckpt[\'model\'])\n        optimizer.load_state_dict(ckpt[\'optimizer\'])\n        scheduler.load_state_dict(ckpt[\'scheduler\'])\n        scaler.load_state_dict(ckpt[\'scaler\'])\n        first, best, history = ckpt[\'epoch\']+1, ckpt[\'best\'], ckpt[\'history\']\n        restore_rng(ckpt[\'rng\'])\n        print(f\'{folder.name}: resume at epoch {first}/{epochs}\', flush=True)\n    targets = {p.stem: strict_label(labels[p.stem], cfg.size) for p in val_paths}\n    for epoch in range(first, epochs+1):\n        begin = time.perf_counter()\n        model.train()\n        losses = []\n        lr = optimizer.param_groups[0][\'lr\']\n        for image, mask in loader:\n            image, mask = image.to(device, non_blocking=True), mask.to(device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, enabled=cfg.amp and device.type==\'cuda\'):\n                loss = segmentation_loss(model(image), mask)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Nonfinite training loss\')\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n            scaler.step(optimizer)\n            scaler.update()\n            losses.append(float(loss.detach()))\n        scheduler.step()\n        score = float(np.mean([dice_score(predict(model,p,cfg,device) >= 0.5, targets[p.stem]) for p in val_paths])) if val_paths else None\n        improved = score is not None and score > best\n        if improved:\n            best = score\n            atomic_checkpoint(folder/\'best.pt\', dict(model=model.state_dict(), epoch=epoch, dice_05=score))\n        row = dict(epoch=epoch, loss=float(np.mean(losses)), dice_05=score, lr=lr,\n                   seconds=round(time.perf_counter()-begin, 3))\n        history.append(row)\n        atomic_checkpoint(last, dict(model=model.state_dict(), optimizer=optimizer.state_dict(),\n                          scheduler=scheduler.state_dict(), scaler=scaler.state_dict(), epoch=epoch,\n                          best=best, history=history, rng=rng_state(), train_ids=[p.stem for p in paths], target_epochs=epochs))\n        with (folder/\'history.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n            writer = csv.DictWriter(f, list(row))\n            writer.writeheader()\n            writer.writerows(history)\n        print(f\'{folder.name} {epoch:03d}/{epochs} loss={row["loss"]:.4f}\' +\n              (f\' val_dice@0.5={score:.5f} best={best:.5f}\' if score is not None else \'\') +\n              f\' seconds={row["seconds"]}\', flush=True)\n    selected = torch.load(folder/(\'best.pt\' if val_paths else \'last.pt\'), map_location=\'cpu\', weights_only=False)\n    model.load_state_dict(selected[\'model\'])\n    return model, int(selected[\'epoch\'])\n\n\ndef rle_encode(mask):\n    flat = np.asarray(mask, dtype=np.uint8).ravel(order=\'C\')\n    changes = np.flatnonzero(np.diff(np.r_[np.uint8(0), flat, np.uint8(0)])) + 1\n    changes[1::2] -= changes[::2]\n    return \' \'.join(map(str, changes))\n\n\ndef rle_decode(encoded, shape=(512,512)):\n    values = np.array([int(s) for s in encoded.split()], dtype=np.int64)\n    if len(values)%2:\n        raise ValueError(\'Odd RLE length\')\n    mask = np.zeros(int(np.prod(shape)), dtype=bool)\n    previous_end = 0\n    for start, length in values.reshape(-1,2):\n        if start < 1 or length < 1 or start <= previous_end or start+length-1 > mask.size:\n            raise ValueError(\'RLE bounds/order/overlap violation\')\n        mask[start-1:start-1+length] = True\n        previous_end = start+length-1\n    return mask.reshape(shape)\n\n\ndef load_converter(root):\n    spec = importlib.util.spec_from_file_location(\'task4_official_converter\', root/\'segmentation_to_csv.py\')\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef preview(paths, probs, labels, threshold, output):\n    tile = 256\n    canvas = Image.new(\'RGB\', (tile*3, (tile+24)*min(4,len(paths))), \'white\')\n    draw = ImageDraw.Draw(canvas)\n    for row,p in enumerate(paths[:4]):\n        rgb = Image.open(p).convert(\'RGB\').resize((tile,tile))\n        target = strict_label(labels[p.stem],tile) if labels else None\n        pred = cv2.resize(probs[p.stem],(tile,tile),interpolation=cv2.INTER_LINEAR) >= threshold\n        y = row*(tile+24)\n        canvas.paste(rgb,(0,y+24))\n        draw.text((4,y+4), f\'Image {p.stem}\',fill=\'black\')\n        if target is not None:\n            canvas.paste(Image.fromarray((target*255).astype(np.uint8)).convert(\'RGB\'),(tile,y+24))\n            draw.text((tile+4,y+4),\'Label: white vessels\',fill=\'black\')\n        else:\n            overlay = np.asarray(rgb).copy()\n            overlay[pred] = (0.4*overlay[pred]+0.6*np.array([0,255,100])).astype(np.uint8)\n            canvas.paste(Image.fromarray(overlay),(tile,y+24))\n            draw.text((tile+4,y+4),\'Vessel overlay\',fill=\'black\')\n        canvas.paste(Image.fromarray((pred*255).astype(np.uint8)).convert(\'RGB\'),(tile*2,y+24))\n        draw.text((tile*2+4,y+4),f\'Prediction t={threshold:.3f}\',fill=\'black\')\n    canvas.save(output)\n\n\ndef validation_report(model, paths, labels, cfg, device, output, epoch):\n    probs = {p.stem:predict(model,p,cfg,device) for p in paths}\n    np.savez_compressed(output/\'validation_probabilities.npz\', **probs)\n    thresholds = [round(float(t),3) for t in np.arange(0.2,0.801,0.025)]\n    rows = []\n    for threshold in thresholds:\n        strict = [dice_score(probs[p.stem]>=threshold, strict_label(labels[p.stem],cfg.size)) for p in paths]\n        proxy = [dice_score(probs[p.stem]>=threshold, converter_label(labels[p.stem])) for p in paths]\n        rows.append(dict(threshold=threshold, mean_vessel_dice=float(np.mean(strict)),\n                         converter_target_proxy_dice=float(np.mean(proxy))))\n    best = max(rows,key=lambda r:(r[\'mean_vessel_dice\'],-abs(r[\'threshold\']-0.5)))\n    threshold = best[\'threshold\']\n    with (output/\'threshold_sweep.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,list(rows[0]));writer.writeheader();writer.writerows(rows)\n    per_image=[]\n    for p in paths:\n        pred=probs[p.stem]>=threshold\n        gt=strict_label(labels[p.stem],cfg.size)\n        native=np.asarray(Image.open(labels[p.stem]).convert(\'L\'))>127\n        native_pred=cv2.resize(probs[p.stem],(native.shape[1],native.shape[0]),interpolation=cv2.INTER_LINEAR)>=threshold\n        per_image.append(dict(id=p.stem,dice_05=dice_score(probs[p.stem]>=0.5,gt),dice=dice_score(pred,gt),\n                              native_dice=dice_score(native_pred,native),\n                              converter_target_proxy_dice=dice_score(pred,converter_label(labels[p.stem])),\n                              predicted_fraction=float(pred.mean()),target_fraction=float(gt.mean())))\n    with (output/\'validation_per_image.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,list(per_image[0]));writer.writeheader();writer.writerows(per_image)\n    report=dict(best_epoch=epoch,threshold=threshold,validation_ids=[p.stem for p in paths],\n                mean_vessel_dice_05=float(np.mean([r[\'dice_05\'] for r in per_image])),\n                mean_vessel_dice=best[\'mean_vessel_dice\'],\n                mean_native_vessel_dice=float(np.mean([r[\'native_dice\'] for r in per_image])),\n                converter_target_proxy_dice=best[\'converter_target_proxy_dice\'],\n                evaluation=\'single held-out fold (4 images), not full five-fold OOF\',\n                limitation=\'Epoch and threshold chosen on the same small validation set; development estimate only.\',\n                private_ground_truth_encoding_verified=False)\n    save_json(output/\'validation_metrics.json\',report)\n    preview(paths,probs,labels,threshold,output/\'validation_preview.png\')\n    return report\n\n\ndef export_submission(model, test, template_ids, cfg, device, root, output, threshold, epoch):\n    white_dir=output/\'masks_512_white_vessels\'\n    black_dir=output/\'masks_native_black_vessels\'\n    white_dir.mkdir(exist_ok=True);black_dir.mkdir(exist_ok=True)\n    probs={p.stem:predict(model,p,cfg,device) for p in test}\n    paths={p.stem:p for p in test}\n    white_paths=[]\n    encoded=[]\n    for image_id in template_ids:\n        prob=probs[image_id]\n        mask=prob>=threshold\n        white_path=white_dir/f\'{image_id}.png\'\n        Image.fromarray((mask*255).astype(np.uint8)).save(white_path)\n        white_paths.append(str(white_path))\n        width,height=Image.open(paths[image_id]).size\n        native=cv2.resize(prob,(width,height),interpolation=cv2.INTER_LINEAR)>=threshold\n        Image.fromarray(np.where(native,0,255).astype(np.uint8)).save(black_dir/f\'{image_id}.png\')\n        value=rle_encode(mask)\n        if not np.array_equal(rle_decode(value),mask):\n            raise AssertionError(\'RLE roundtrip mismatch\')\n        encoded.append(value)\n    # Execute the supplied conversion function on the actual PNG predictions.\n    official=load_converter(root).turn_to_str(white_paths)\n    if encoded != official:\n        raise AssertionError(\'Vectorized encoding differs from the official converter\')\n    target=output/\'submission.csv\'\n    with target.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.writer(f);writer.writerow([\'Id\',\'Predicted\']);writer.writerows(zip(template_ids,official))\n    with target.open(newline=\'\',encoding=\'utf-8\') as f:\n        rows=list(csv.DictReader(f))\n    if [r[\'Id\'] for r in rows]!=template_ids or len(rows)!=20:\n        raise AssertionError(\'Saved submission IDs/count mismatch\')\n    for row in rows:\n        rle_decode(row[\'Predicted\'])\n    np.savez_compressed(output/\'test_probabilities.npz\',**probs)\n    preview(test,probs,None,threshold,output/\'test_preview.png\')\n    save_json(output/\'submission_provenance.json\',dict(\n        csv_sha256=file_sha(target),model_sha256=file_sha(output/\'fulltrain/last.pt\'),\n        train_images=20,epochs=epoch,threshold=threshold,external_training_data=False,pretrained=False,\n        csv_polarity=\'white vessels, following observed training labels and supplied converter\',\n        image_output_polarity=\'masks_native_black_vessels: vessel=0, background=255\',\n        official_converter_exact_match=True,rle_roundtrip_all_20=True,template_order_preserved=True,\n        submitted_to_competition=False))\n    print(f\'Submission saved: {target}\',flush=True)\n    return target\n\n\ndef run(phase=\'full\', data_dir=None, output_dir=None, config=None):\n    cfg=config or Config()\n    if cfg.size!=512 or cfg.patch<16 or cfg.patch%16 or cfg.patch>cfg.size or cfg.base_channels<8 or cfg.base_channels%8:\n        raise ValueError(\'Require size=512, patch divisible by 16 <=512, base_channels divisible by 8.\')\n    if cfg.epochs<1 or cfg.patches_per_epoch<1 or cfg.batch_size<1 or cfg.threads<1 or not 2<=cfg.folds<=20 or not 0<=cfg.fold<cfg.folds:\n        raise ValueError(\'Invalid training configuration\')\n    torch.set_num_threads(cfg.threads)\n    cv2.setNumThreads(0)\n    root=find_data(data_dir)\n    output=Path(output_dir or \'task4_baseline_results\').resolve()\n    output.mkdir(parents=True,exist_ok=True)\n    source_sha=file_sha(Path(__file__)) if \'__file__\' in globals() else PIPELINE_SHA256\n    config_path=output/\'run_config.json\'\n    previous=json.loads(config_path.read_text(encoding=\'utf-8\')) if config_path.exists() else None\n    if previous and (previous[\'config\']!=asdict(cfg) or previous[\'pipeline_sha256\']!=source_sha):\n        raise ValueError(\'Existing output belongs to another config/source. Use a new directory.\')\n    train,test,labels,assignment,ids,summary=audit(root,output,cfg)\n    identity=dict(config=asdict(cfg),data_fingerprint=summary[\'data_fingerprint\'],\n                  pipeline_sha256=source_sha)\n    if previous and previous!=identity:\n        raise ValueError(\'Existing output belongs to another config/data/source. Use a new directory.\')\n    save_json(config_path,identity)\n    if phase==\'audit\':\n        print(json.dumps(summary,ensure_ascii=False,indent=2),flush=True)\n        return output\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    print(f\'Device: {device}; train=16, validation=4; no pretrained weights\',flush=True)\n    val=[p for p in train if assignment[p.stem]==cfg.fold]\n    development=[p for p in train if assignment[p.stem]!=cfg.fold]\n    # A single writer protects checkpoints from two simultaneous runs.\n    lock=output/\'training.lock\'\n    try:\n        descriptor=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)\n    except FileExistsError:\n        raise RuntimeError(f\'Run lock exists: {lock}. Check its PID before removing a stale lock.\')\n    os.write(descriptor,str(os.getpid()).encode());os.close(descriptor)\n    try:\n        model,epoch=train_stage(development,val,labels,cfg,output/\'holdout\',device,cfg.epochs)\n        metrics=validation_report(model,val,labels,cfg,device,output,epoch)\n        print(json.dumps(metrics,ensure_ascii=False,indent=2),flush=True)\n        if phase==\'full\':\n            del model\n            if device.type==\'cuda\':torch.cuda.empty_cache()\n            model,full_epoch=train_stage(train,[],labels,cfg,output/\'fulltrain\',device,epoch)\n            export_submission(model,test,ids,cfg,device,root,output,metrics[\'threshold\'],full_epoch)\n    finally:\n        lock.unlink(missing_ok=True)\n    return output\n\n\ndef cli():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--phase\',choices=[\'audit\',\'train\',\'full\'],default=\'full\')\n    parser.add_argument(\'--data-dir\')\n    parser.add_argument(\'--output-dir\',default=\'task4_baseline_results\')\n    for key,value in asdict(Config()).items():\n        if isinstance(value,bool):continue\n        parser.add_argument(\'--\'+key.replace(\'_\',\'-\'),type=type(value),default=value)\n    args=vars(parser.parse_args())\n    phase,data_dir,output=args.pop(\'phase\'),args.pop(\'data_dir\'),args.pop(\'output_dir\')\n    run(phase,data_dir,output,Config(**args))\n\n\nif __name__==\'__main__\':\n    cli()\n'
V2_SOURCE = '"""Fixed-epoch five-fold ResNet34 U-Net; native pixels, EMA, paired D4 evaluation.\n\nThe notebook embeds this file AND task4_pipeline.py, so no extra source upload is needed.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport shutil\nimport socket\nimport time\nimport zipfile\nfrom contextlib import contextmanager\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nimport torchvision\nfrom PIL import Image\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision.models import ResNet34_Weights, resnet34\n\nimport task4_pipeline as base\n\n\n@dataclass\nclass ConfigV2:\n    seed: int = 42\n    folds: int = 5\n    fold: int = 0                 # audit helper only; run_cv uses all requested folds\n    epochs: int = 80\n    patch: int = 256\n    patches_per_epoch: int = 256\n    batch_size: int = 8\n    encoder_lr: float = 0.0001\n    decoder_lr: float = 0.0007\n    weight_decay: float = 0.0001\n    warmup_epochs: int = 3\n    ema_decay: float = 0.99\n    monitor_every: int = 10\n    threads: int = 4\n    amp: bool = True\n    pretrained: bool = True\n    tta_gain_required: float = 0.001\n\n\ndef audit_v2(root, output, cfg):\n    """Freeze the original ID folds; JPEG decoder changes must not reshuffle folds."""\n    train,test,labels,_,ids,summary=base.audit(root,output,cfg)\n    frozen=[[\'2\',\'6\',\'9\',\'10\'],[\'3\',\'14\',\'17\',\'20\'],[\'1\',\'4\',\'5\',\'11\'],\n            [\'15\',\'16\',\'18\',\'19\'],[\'7\',\'8\',\'12\',\'13\']]\n    assignment={i:fold for fold,group in enumerate(frozen) for i in group}\n    if set(assignment)!={p.stem for p in train}:\n        raise ValueError(\'This frozen task-4 split expects train IDs 1 through 20.\')\n    manifest=output/\'manifest.csv\'\n    with manifest.open(newline=\'\',encoding=\'utf-8\') as f:\n        reader=csv.DictReader(f);fields=reader.fieldnames;rows=list(reader)\n    groups={}\n    for row in rows:\n        if row[\'split\']==\'train\':\n            row[\'fold\']=assignment[row[\'id\']]\n            groups.setdefault(row[\'pixel_sha256\'],set()).add(row[\'fold\'])\n    if any(len(folds)>1 for folds in groups.values()):\n        raise ValueError(\'A decoded duplicate crosses frozen folds; inspect dataset before training.\')\n    with manifest.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,fields);writer.writeheader();writer.writerows(rows)\n    # Raw file hashes are stable across JPEG decoder/library versions, unlike pixel hashes.\n    stable=[{k:row[k] for k in [\'split\',\'id\',\'width\',\'height\',\'image_sha256\',\'label_sha256\',\'fold\']}\n            for row in rows]\n    summary.update(data_fingerprint=hashlib.sha256(json.dumps(stable,sort_keys=True).encode()).hexdigest(),\n                   validation_ids=frozen[0],folds=[4]*5,\n                   split_protocol=\'frozen original seed42 ID folds; training seed may vary\')\n    base.save_json(output/\'audit_summary.json\',summary)\n    return train,test,labels,assignment,ids,summary\n\n\ndef read_rgb(path):\n    return np.asarray(Image.open(path).convert(\'RGB\')).copy()\n\n\ndef read_mask(path):\n    return (np.asarray(Image.open(path).convert(\'L\')) > 127).astype(np.float32)\n\n\ndef resize_probability(prob, shape=(512, 512)):\n    # PIL receives a float image: interpolate probabilities, then threshold once.\n    h, w = shape\n    return np.clip(np.asarray(Image.fromarray(np.asarray(prob, np.float32)).resize(\n        (w, h), Image.Resampling.BILINEAR)), 0, 1).copy()\n\n\nclass NativePatches(Dataset):\n    def __init__(self, paths, labels, cfg):\n        self.images = [read_rgb(p) for p in paths]\n        self.masks = [read_mask(labels[p.stem]) for p in paths]\n        self.positives = [np.argwhere(m > 0) for m in self.masks]\n        self.cfg = cfg\n\n    def __len__(self):\n        return self.cfg.patches_per_epoch\n\n    def __getitem__(self, index):\n        i = random.randrange(len(self.images))\n        image, mask = self.images[i], self.masks[i]\n        h, w = mask.shape\n        patch = self.cfg.patch\n        crop = random.randint(int(patch*0.9), min(int(patch*1.1), h, w))\n        if random.random() < 0.3 and len(self.positives[i]):\n            y, x = self.positives[i][random.randrange(len(self.positives[i]))]\n            top, left = int(np.clip(y-crop//2, 0, h-crop)), int(np.clip(x-crop//2, 0, w-crop))\n        else:\n            top, left = random.randint(0, h-crop), random.randint(0, w-crop)\n        image = cv2.resize(image[top:top+crop,left:left+crop], (patch,patch), interpolation=cv2.INTER_LINEAR)\n        mask = cv2.resize(mask[top:top+crop,left:left+crop], (patch,patch), interpolation=cv2.INTER_NEAREST)\n        image, mask = base.paired_d4(image,mask,random.randrange(4),random.random()<0.5)\n        image = image.astype(np.float32)/255\n        image = np.clip((image-0.5)*random.uniform(0.85,1.15)+0.5+random.uniform(-0.04,0.04),0,1)\n        image = image**random.uniform(0.85,1.15)\n        image = np.clip(image*np.array([random.uniform(0.95,1.05) for _ in range(3)],np.float32),0,1)\n        return torch.from_numpy(image.transpose(2,0,1).copy()), torch.from_numpy(mask[None].copy())\n\n\nclass ResNet34UNet(nn.Module):\n    def __init__(self, initial_state=None):\n        super().__init__()\n        self.encoder = resnet34(weights=None)\n        if initial_state is not None:\n            self.encoder.load_state_dict(initial_state,strict=True)\n        self.encoder.fc = nn.Identity()\n        self.shallow = base.ConvBlock(3,32)\n        self.up4 = base.ConvBlock(512+256,256)\n        self.up3 = base.ConvBlock(256+128,128)\n        self.up2 = base.ConvBlock(128+64,64)\n        self.up1 = base.ConvBlock(64+64,48)\n        self.up0 = base.ConvBlock(48+32,32)\n        self.head = nn.Conv2d(32,1,1)\n        self.aux2 = nn.Conv2d(48,1,1)\n        self.aux4 = nn.Conv2d(64,1,1)\n        self.register_buffer(\'mean\',torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))\n        self.register_buffer(\'std\',torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))\n\n    def train(self, mode=True):\n        super().train(mode)\n        # Small batches: keep ImageNet running statistics; affine parameters still train.\n        for module in self.encoder.modules():\n            if isinstance(module,nn.BatchNorm2d):\n                module.eval()\n        return self\n\n    @staticmethod\n    def join(x, skip, block):\n        x = F.interpolate(x,size=skip.shape[-2:],mode=\'bilinear\',align_corners=False)\n        return block(torch.cat([x,skip],1))\n\n    def forward(self, image):\n        x = (image-self.mean)/self.std\n        s0 = self.shallow(x)\n        s1 = self.encoder.relu(self.encoder.bn1(self.encoder.conv1(x)))\n        s2 = self.encoder.layer1(self.encoder.maxpool(s1))\n        s3 = self.encoder.layer2(s2)\n        s4 = self.encoder.layer3(s3)\n        z = self.encoder.layer4(s4)\n        z = self.join(z,s4,self.up4)\n        z = self.join(z,s3,self.up3)\n        z4 = self.join(z,s2,self.up2)\n        z2 = self.join(z4,s1,self.up1)\n        z = self.join(z2,s0,self.up0)\n        return self.head(z),self.aux2(z2),self.aux4(z4)\n\n\ndef objective(outputs, target):\n    losses = []\n    for logits in outputs:\n        truth = target if logits.shape[-2:]==target.shape[-2:] else F.interpolate(\n            target,size=logits.shape[-2:],mode=\'area\')\n        losses.append(base.segmentation_loss(logits,truth))\n    return (losses[0]+0.1*losses[1]+0.2*losses[2])/1.3\n\n\n@torch.no_grad()\ndef update_ema(ema, model, decay):\n    for target, source in zip(ema.parameters(),model.parameters()):\n        target.lerp_(source,1-decay)\n    for target, source in zip(ema.buffers(),model.buffers()):\n        target.copy_(source)\n\n\ndef transform_d4(x, rotation, flip):\n    x = torch.rot90(x,rotation,(-2,-1))\n    return torch.flip(x,(-1,)) if flip else x\n\n\ndef inverse_d4(x, rotation, flip):\n    if flip:\n        x = torch.flip(x,(-1,))\n    return torch.rot90(x,-rotation,(-2,-1))\n\n\ndef window_starts(length, patch, stride):\n    if length < patch:\n        raise ValueError(\'Pad before calculating window starts\')\n    starts = list(range(0,length-patch+1,stride))\n    if starts[-1]!=length-patch:\n        starts.append(length-patch)\n    return starts\n\n\n@torch.inference_mode()\ndef predict_pair(model, image, cfg, device, with_tta=True):\n    """Patchwise D4; return native probabilities for none and D4 using identical windows."""\n    model.eval()\n    if isinstance(image,(str,Path)):\n        image = read_rgb(image)\n    h,w = image.shape[:2]\n    pad_h,pad_w = max(0,cfg.patch-h),max(0,cfg.patch-w)\n    image = cv2.copyMakeBorder(image,0,pad_h,0,pad_w,cv2.BORDER_REFLECT_101)\n    height,width = image.shape[:2]\n    coords = [(y,x) for y in window_starts(height,cfg.patch,cfg.patch//2)\n              for x in window_starts(width,cfg.patch,cfg.patch//2)]\n    weight = np.maximum(np.outer(np.hanning(cfg.patch),np.hanning(cfg.patch)),0.05).astype(np.float32)\n    accum = np.zeros((2,height,width),np.float32)\n    weights = np.zeros((height,width),np.float32)\n    transforms = [(0,False)] + ([(r,f) for r in range(4) for f in [False,True] if (r,f)!=(0,False)] if with_tta else [])\n    for start in range(0,len(coords),cfg.batch_size):\n        positions = coords[start:start+cfg.batch_size]\n        batch = np.stack([image[y:y+cfg.patch,x:x+cfg.patch].transpose(2,0,1) for y,x in positions])\n        tensor = torch.from_numpy(batch.copy()).to(device).float()/255\n        plain,total = None,None\n        for rotation,flip in transforms:\n            transformed = transform_d4(tensor,rotation,flip)\n            with torch.autocast(device_type=device.type,enabled=cfg.amp and device.type==\'cuda\'):\n                prediction = model(transformed)[0].float().sigmoid()\n            prediction = inverse_d4(prediction,rotation,flip)\n            if plain is None:\n                plain = prediction\n                total = prediction.clone()\n            else:\n                total.add_(prediction)\n        pair = torch.stack([plain,total/len(transforms)])[:,:,0].cpu().numpy()\n        for i,(y,x) in enumerate(positions):\n            accum[:,y:y+cfg.patch,x:x+cfg.patch] += pair[:,i]*weight\n            weights[y:y+cfg.patch,x:x+cfg.patch] += weight\n    result = accum/weights[None]\n    if not np.isfinite(result).all():\n        raise FloatingPointError(\'Nonfinite inference output\')\n    return {\'none\':result[0,:h,:w].copy(),\'d4\':result[1,:h,:w].copy()}\n\n\ndef resolve_weights(cfg, output, weights_path=None):\n    if not cfg.pretrained:\n        return None,dict(source=\'random initialization; debug/ablation only\',sha256=None)\n    enum = ResNet34_Weights.IMAGENET1K_V1\n    target = Path(weights_path) if weights_path else output/\'pretrained_cache\'/Path(enum.url).name\n    if not target.is_file():\n        if weights_path:\n            raise FileNotFoundError(target)\n        target.parent.mkdir(parents=True,exist_ok=True)\n        print(\'Downloading official ImageNet encoder weights:\',enum.url,flush=True)\n        torch.hub.download_url_to_file(enum.url,str(target),hash_prefix=\'b627a593\',progress=True)\n    digest = base.file_sha(target)\n    if not digest.startswith(\'b627a593\'):\n        raise ValueError(\'Expected official torchvision resnet34-b627a593.pth, not task-finetuned weights.\')\n    state = torch.load(target,map_location=\'cpu\',weights_only=True)\n    probe = resnet34(weights=None)\n    probe.load_state_dict(state,strict=True)\n    del probe\n    cached=output/\'pretrained_cache\'/Path(enum.url).name\n    if target.resolve()!=cached.resolve():\n        cached.parent.mkdir(parents=True,exist_ok=True)\n        shutil.copyfile(target,cached)\n    return state,dict(source=enum.url,sha256=digest,supervision=\'generic ImageNet classification, not vessel segmentation\')\n\n\ndef set_lrs(optimizer, step, steps_per_epoch, cfg):\n    total = cfg.epochs*steps_per_epoch\n    warm = min(cfg.warmup_epochs*steps_per_epoch,max(0,total-1))\n    if warm and step<warm:\n        factor = (step+1)/warm\n    else:\n        progress = (step-warm)/max(1,total-warm-1)\n        factor = 0.02+0.98*0.5*(1+math.cos(math.pi*min(1,progress)))\n    for group,initial in zip(optimizer.param_groups,[cfg.encoder_lr,cfg.decoder_lr]):\n        group[\'lr\'] = initial*factor\n\n\ndef train_fold(paths, val_paths, labels, cfg, fold, folder, initial_state, device):\n    folder.mkdir(parents=True,exist_ok=True)\n    base.seed_all(cfg.seed+1009*fold)\n    dataset = NativePatches(paths,labels,cfg)\n    loader = DataLoader(dataset,batch_size=cfg.batch_size,num_workers=0,shuffle=False,\n                        pin_memory=device.type==\'cuda\',drop_last=False)\n    model = ResNet34UNet(initial_state).to(device)\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    encoder = list(model.encoder.parameters())\n    decoder = [p for name,p in model.named_parameters() if not name.startswith(\'encoder.\')]\n    optimizer = torch.optim.AdamW([{\'params\':encoder,\'lr\':cfg.encoder_lr},\n                                 {\'params\':decoder,\'lr\':cfg.decoder_lr}],weight_decay=cfg.weight_decay)\n    scaler = torch.amp.GradScaler(\'cuda\',enabled=cfg.amp and device.type==\'cuda\')\n    history,first,step = [],1,0\n    train_ids,val_ids = [p.stem for p in paths],[p.stem for p in val_paths]\n    last = folder/\'last.pt\'\n    if last.exists():\n        ckpt = torch.load(last,map_location=\'cpu\',weights_only=False)\n        if ckpt[\'train_ids\']!=train_ids or ckpt[\'val_ids\']!=val_ids:\n            raise ValueError(\'Checkpoint split mismatch\')\n        model.load_state_dict(ckpt[\'model\']);ema.load_state_dict(ckpt[\'ema\'])\n        optimizer.load_state_dict(ckpt[\'optimizer\']);scaler.load_state_dict(ckpt[\'scaler\'])\n        history,first,step = ckpt[\'history\'],ckpt[\'epoch\']+1,ckpt[\'step\']\n        base.restore_rng(ckpt[\'rng\'])\n        del ckpt\n        print(f\'fold{fold}: resume at {first}/{cfg.epochs}\',flush=True)\n    for epoch in range(first,cfg.epochs+1):\n        begin = time.perf_counter()\n        model.train()\n        losses = []\n        for image,target in loader:\n            set_lrs(optimizer,step,len(loader),cfg)\n            image,target = image.to(device,non_blocking=True),target.to(device,non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type,enabled=cfg.amp and device.type==\'cuda\'):\n                loss = objective(model(image),target)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Nonfinite training loss\')\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(model.parameters(),5.0)\n            old_scale = scaler.get_scale()\n            scaler.step(optimizer);scaler.update()\n            if scaler.get_scale()>=old_scale:\n                update_ema(ema,model,cfg.ema_decay)\n            losses.append(float(loss.detach()))\n            step += 1\n        score = None\n        if epoch%cfg.monitor_every==0 or epoch==cfg.epochs:\n            score = float(np.mean([base.dice_score(\n                resize_probability(predict_pair(ema,p,cfg,device,False)[\'none\'])>=0.5,\n                base.strict_label(labels[p.stem])) for p in val_paths]))\n        row = dict(epoch=epoch,loss=float(np.mean(losses)),monitor_dice_05=score,\n                   encoder_lr=optimizer.param_groups[0][\'lr\'],decoder_lr=optimizer.param_groups[1][\'lr\'],\n                   seconds=round(time.perf_counter()-begin,3))\n        history.append(row)\n        if shutil.disk_usage(folder).free<1200*1024**2:\n            raise OSError(\'Less than 1.2 GiB free: preserve old checkpoint and free space before resuming.\')\n        base.atomic_checkpoint(last,dict(model=model.state_dict(),ema=ema.state_dict(),\n            optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),history=history,\n            epoch=epoch,step=step,rng=base.rng_state(),train_ids=train_ids,val_ids=val_ids))\n        with (folder/\'history.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            writer=csv.DictWriter(f,list(row));writer.writeheader();writer.writerows(history)\n        print(f\'fold{fold} {epoch:03d}/{cfg.epochs} loss={row["loss"]:.4f} \'+\n              (f\'monitor_dice@0.5={score:.5f} \' if score is not None else \'\')+\n              f\'seconds={row["seconds"]}\',flush=True)\n        if epoch==1:\n            print(f\'fold{fold}: measured estimate for this fold training/monitoring: \'\n                  f\'{row["seconds"]*cfg.epochs/60:.1f} min (startup may inflate this; D4 export extra)\',flush=True)\n    # Final EMA is fixed in advance. Validation never selects epochs/checkpoints.\n    base.atomic_checkpoint(folder/\'final_ema.pt\',dict(model=ema.state_dict(),epoch=cfg.epochs,\n                          train_ids=train_ids,val_ids=val_ids,selection=\'predeclared final EMA\'))\n    del model,optimizer,scaler,dataset,loader\n    if device.type==\'cuda\':torch.cuda.empty_cache()\n    return ema\n\n\ndef infer_fold(model, val_paths, test, labels, cfg, folder, device):\n    val_probs,test_probs = {},{}\n    for p in val_paths:\n        pair = predict_pair(model,p,cfg,device)\n        for mode,native in pair.items():\n            val_probs[f\'{mode}_{p.stem}\'] = resize_probability(native)\n            val_probs[f\'native_{mode}_{p.stem}\'] = native\n    np.savez_compressed(folder/\'validation_probabilities.npz\',**val_probs)\n    for p in test:\n        pair = predict_pair(model,p,cfg,device)\n        for mode,native in pair.items():\n            test_probs[f\'{mode}_{p.stem}\'] = resize_probability(native)\n    np.savez_compressed(folder/\'test_probabilities.npz\',**test_probs)\n    clean = {p.stem:base.strict_label(labels[p.stem]) for p in val_paths}\n    summary = dict(epoch=cfg.epochs,val_ids=[p.stem for p in val_paths],\n        none_dice_05=float(np.mean([base.dice_score(val_probs[f\'none_{i}\']>=0.5,clean[i]) for i in clean])),\n        d4_dice_05=float(np.mean([base.dice_score(val_probs[f\'d4_{i}\']>=0.5,clean[i]) for i in clean])))\n    files=[\'history.csv\',\'last.pt\',\'final_ema.pt\',\'validation_probabilities.npz\',\'test_probabilities.npz\']\n    summary[\'sha256\']={name:base.file_sha(folder/name) for name in files}\n    base.save_json(folder/\'completed.json\',summary)\n    print(\'Fold completed:\',summary[\'val_ids\'],summary[\'none_dice_05\'],summary[\'d4_dice_05\'],flush=True)\n\n\ndef threshold_sweep(probs, targets):\n    thresholds = np.round(np.arange(0.1,0.901,0.025),3)\n    rows=[dict(threshold=float(t),mean_dice=float(np.mean([\n        base.dice_score(probs[i]>=t,targets[i]) for i in probs]))) for t in thresholds]\n    best=max(rows,key=lambda r:(r[\'mean_dice\'],-abs(r[\'threshold\']-0.5)))\n    return best,rows\n\n\ndef export_csv(probs, ids, threshold, name, root, output):\n    folder=output/\'masks_512_white_vessels\'/name\n    folder.mkdir(parents=True,exist_ok=True)\n    filenames,encoded=[],[]\n    for i in ids:\n        mask=probs[i]>=threshold\n        path=folder/f\'{i}.png\';Image.fromarray((mask*255).astype(np.uint8)).save(path)\n        value=base.rle_encode(mask)\n        if not np.array_equal(base.rle_decode(value),mask):\n            raise AssertionError(\'RLE mismatch\')\n        filenames.append(str(path));encoded.append(value)\n    official=base.load_converter(root).turn_to_str(filenames)\n    if official!=encoded:raise AssertionError(\'Official converter mismatch\')\n    path=output/f\'{name}.csv\'\n    with path.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.writer(f);writer.writerow([\'Id\',\'Predicted\']);writer.writerows(zip(ids,official))\n    return dict(file=path.name,threshold=threshold,sha256=base.file_sha(path),\n                rows=len(ids),official_converter_exact_match=True,rle_roundtrip=True)\n\n\ndef aggregate(root, output, cfg, train, test, labels, assignment, ids):\n    folds=[output/f\'fold{f}\' for f in range(cfg.folds)]\n    if not all((folder/\'completed.json\').exists() for folder in folds):\n        finished=[f for f,folder in enumerate(folds) if (folder/\'completed.json\').exists()]\n        base.save_json(output/\'partial_cv_summary.json\',dict(completed_folds=finished,complete_oof=False))\n        print(\'Partial CV only; no full-OOF score or ensemble submission. Completed:\',finished,flush=True)\n        return None\n    oof={mode:{} for mode in [\'none\',\'d4\']}\n    native={mode:{} for mode in [\'none\',\'d4\']}\n    test_mean={mode:{i:np.zeros((512,512),np.float64) for i in ids} for mode in oof}\n    for fold,folder in enumerate(folds):\n        metadata=json.loads((folder/\'completed.json\').read_text(encoding=\'utf-8\'))\n        for name,digest in metadata[\'sha256\'].items():\n            if base.file_sha(folder/name)!=digest:raise ValueError(f\'Artifact changed: {folder/name}\')\n        expected={p.stem for p in train if assignment[p.stem]==fold}\n        if set(metadata[\'val_ids\'])!=expected:raise ValueError(\'Validation IDs mismatch\')\n        with np.load(folder/\'validation_probabilities.npz\') as values:\n            for mode in oof:\n                for i in expected:\n                    if i in oof[mode]:raise AssertionError(\'Repeated OOF ID\')\n                    oof[mode][i]=values[f\'{mode}_{i}\'].copy()\n                    native[mode][i]=values[f\'native_{mode}_{i}\'].copy()\n        with np.load(folder/\'test_probabilities.npz\') as values:\n            for mode in oof:\n                for i in ids:test_mean[mode][i]+=values[f\'{mode}_{i}\']/cfg.folds\n    train_ids={p.stem for p in train}\n    if any(set(probs)!=train_ids for probs in oof.values()):raise AssertionError(\'Incomplete OOF coverage\')\n    clean={i:base.strict_label(labels[i]) for i in train_ids}\n    proxy={i:base.converter_label(labels[i]) for i in train_ids}\n    rows,mode_summary,exported=[],{},{}\n    for mode in oof:\n        calibrated,sweep=threshold_sweep(oof[mode],clean)\n        proxy_calibrated,proxy_sweep=threshold_sweep(oof[mode],proxy)\n        threshold=calibrated[\'threshold\']\n        cross_thresholds={}\n        for fold in range(cfg.folds):\n            other={j:oof[mode][j] for j in train_ids if assignment[j]!=fold}\n            cross_thresholds[fold]=threshold_sweep(other,{j:clean[j] for j in other})[0][\'threshold\']\n        with (output/f\'threshold_sweep_{mode}.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            writer=csv.DictWriter(f,[\'threshold\',\'clean_dice\',\'converter_proxy_dice\']);writer.writeheader()\n            writer.writerows(dict(threshold=a[\'threshold\'],clean_dice=a[\'mean_dice\'],converter_proxy_dice=b[\'mean_dice\'])\n                             for a,b in zip(sweep,proxy_sweep))\n        per=[]\n        for i in sorted(train_ids,key=int):\n            # Threshold calibration excluding this outer fold: diagnostic, not nested CV.\n            cross_t=cross_thresholds[assignment[i]]\n            gt_native=read_mask(labels[i])>0\n            row=dict(id=i,fold=assignment[i],mode=mode,\n                dice_05=base.dice_score(oof[mode][i]>=0.5,clean[i]),\n                dice_global_threshold=base.dice_score(oof[mode][i]>=threshold,clean[i]),\n                dice_other_fold_threshold=base.dice_score(oof[mode][i]>=cross_t,clean[i]),\n                other_fold_threshold=cross_t,\n                native_dice_05=base.dice_score(native[mode][i]>=0.5,gt_native),\n                converter_proxy_dice=base.dice_score(oof[mode][i]>=threshold,proxy[i]))\n            rows.append(row);per.append(row)\n        mode_summary[mode]=dict(mean_dice_05=float(np.mean([r[\'dice_05\'] for r in per])),\n            global_threshold=threshold,mean_dice_global_threshold=calibrated[\'mean_dice\'],\n            mean_dice_other_fold_threshold=float(np.mean([r[\'dice_other_fold_threshold\'] for r in per])),\n            mean_native_dice_05=float(np.mean([r[\'native_dice_05\'] for r in per])),\n            mean_converter_proxy_dice=float(np.mean([r[\'converter_proxy_dice\'] for r in per])),\n            converter_proxy_best_threshold=proxy_calibrated[\'threshold\'],\n            converter_proxy_best_dice=proxy_calibrated[\'mean_dice\'])\n        exported[mode]=export_csv(test_mean[mode],ids,threshold,f\'submission_{mode}\',root,output)\n    chosen=\'d4\' if mode_summary[\'d4\'][\'mean_dice_05\']>=mode_summary[\'none\'][\'mean_dice_05\']+cfg.tta_gain_required else \'none\'\n    source=output/exported[chosen][\'file\']\n    shutil.copyfile(source,output/\'submission.csv\')\n    # Explicitly named diagnostic candidate; never silently substitute it for clean vessels.\n    proxy_file=export_csv(test_mean[chosen],ids,mode_summary[chosen][\'converter_proxy_best_threshold\'],\n                         \'submission_converter_proxy\',root,output)\n    threshold=mode_summary[chosen][\'global_threshold\']\n    base.preview(test,{p.stem:test_mean[chosen][p.stem] for p in test},None,threshold,output/\'test_preview.png\')\n    fold0=[p for p in train if assignment[p.stem]==0]\n    base.preview(fold0,oof[chosen],labels,threshold,output/\'oof_fold0_preview.png\')\n    black=output/\'masks_native_black_vessels\';black.mkdir(exist_ok=True)\n    for p in test:\n        width,height=Image.open(p).size\n        prob=resize_probability(test_mean[chosen][p.stem],(height,width))\n        Image.fromarray(np.where(prob>=threshold,0,255).astype(np.uint8)).save(black/f\'{p.stem}.png\')\n    with (output/\'oof_per_image.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,list(rows[0]));writer.writeheader();writer.writerows(rows)\n    np.savez_compressed(output/\'oof_probabilities.npz\',**{f\'{mode}_{i}\':p for mode,probs in oof.items() for i,p in probs.items()})\n    np.savez_compressed(output/\'test_ensemble_probabilities.npz\',**{f\'{mode}_{i}\':p.astype(np.float32) for mode,probs in test_mean.items() for i,p in probs.items()})\n    summary=dict(complete_oof=True,unique_oof_images=len(train_ids),epochs=cfg.epochs,\n        checkpoint_selection=\'fixed final EMA; no outer-validation epoch selection\',\n        modes=mode_summary,selected_mode=chosen,tta_gain_required=cfg.tta_gain_required,\n        test_ensemble=\'equal average of five fold models; each trained on 16 images\',\n        baseline_reference=dict(ids=[\'2\',\'6\',\'9\',\'10\'],baseline_dice_05=0.7839949354578135,\n            v2_same_ids_none_dice_05=float(np.mean([r[\'dice_05\'] for r in rows if r[\'id\'] in {\'2\',\'6\',\'9\',\'10\'} and r[\'mode\']==\'none\'])),\n            note=\'Matched subset only; baseline did not produce full five-fold OOF.\'),\n        selection_caveat=\'TTA mode and global threshold use OOF labels. Tuned scores are development estimates. \'\n                         \'Other-fold calibration is diagnostic, not a nested independent test.\',\n        private_target_preprocessing_verified=False,submitted_to_competition=False)\n    base.save_json(output/\'cv_summary.json\',summary)\n    base.save_json(output/\'submission_provenance.json\',dict(chosen_mode=chosen,\n        primary_csv_sha256=base.file_sha(output/\'submission.csv\'),alternatives=exported,\n        converter_proxy_candidate=proxy_file,fold_models=[base.file_sha(p/\'final_ema.pt\') for p in folds],\n        calibrated_on=\'20 OOF clean masks; proxy candidate separately calibrated on JPEG converter targets\',\n        image_polarity=\'CSV white vessels; separate native-size PNGs have vessel=0, background=255\',\n        official_converter_exact_match=True,external_vessel_training_data=False))\n    print(json.dumps(summary,ensure_ascii=False,indent=2),flush=True)\n    return summary\n\n\ndef make_backups(output, full=True):\n    """Portable analysis archive; optional full archive includes model and resume states."""\n    output=Path(output)\n    files=[p for p in output.rglob(\'*\') if p.is_file() and p.suffix not in {\'.zip\',\'.tmp\'} and p.name!=\'training.lock\']\n    light=[p for p in files if p.suffix in {\'.json\',\'.csv\',\'.png\',\'.py\',\'.md\'} and \'pretrained_cache\' not in p.parts\n           and not (p.suffix==\'.png\' and \'masks_512_white_vessels\' in p.parts)]\n    for name,paths in [(\'analysis_results.zip\',light)]+([(\'full_backup.zip\',files)] if full else []):\n        target=output/name;tmp=output/(name+\'.tmp\')\n        estimated=sum(p.stat().st_size for p in paths)\n        if shutil.disk_usage(output).free<estimated+256*1024**2:\n            raise OSError(f\'Not enough free space to create {name}; completed results remain intact.\')\n        with zipfile.ZipFile(tmp,\'w\',compression=zipfile.ZIP_STORED,allowZip64=True) as z:\n            for path in paths:z.write(path,path.relative_to(output).as_posix())\n        with zipfile.ZipFile(tmp) as z:\n            bad=z.testzip()\n            if bad:raise IOError(f\'Archive CRC failed: {bad}\')\n        os.replace(tmp,target)\n        print(\'Backup saved:\',target,f\'{target.stat().st_size/1024**2:.1f} MiB\',flush=True)\n\n\ndef restore_output(source, output):\n    source,output=Path(source).resolve(),Path(output).resolve()\n    if source==output:return\n    if output.exists() and any(output.iterdir()):\n        # Existing working output is the authoritative continuation; do not overlay backup.\n        if (output/\'run_config.json\').exists():return\n        raise ValueError(\'Restore target is not empty\')\n    output.mkdir(parents=True,exist_ok=True)\n    if source.is_dir():\n        shutil.copytree(source,output,dirs_exist_ok=True,ignore=shutil.ignore_patterns(\'*.zip\',\'*.tmp\',\'training.lock\'))\n    elif source.suffix.lower()==\'.zip\':\n        with zipfile.ZipFile(source) as z:\n            for item in z.infolist():\n                resolved=(output/item.filename).resolve()\n                if not resolved.is_relative_to(output):raise ValueError(\'Unsafe archive path\')\n            z.extractall(output)\n        (output/\'training.lock\').unlink(missing_ok=True)\n    else:raise ValueError(\'RESUME_FROM must be an existing output folder or full_backup.zip\')\n\n\n@contextmanager\ndef run_lock(output):\n    lock=output/\'training.lock\'\n    if lock.exists():\n        info=json.loads(lock.read_text())\n        alive=False\n        if info.get(\'host\')==socket.gethostname():\n            if os.name==\'nt\':\n                import ctypes\n                from ctypes import wintypes\n                kernel=ctypes.WinDLL(\'kernel32\',use_last_error=True)\n                kernel.OpenProcess.argtypes=[wintypes.DWORD,wintypes.BOOL,wintypes.DWORD]\n                kernel.OpenProcess.restype=wintypes.HANDLE\n                kernel.GetExitCodeProcess.argtypes=[wintypes.HANDLE,ctypes.POINTER(wintypes.DWORD)]\n                kernel.CloseHandle.argtypes=[wintypes.HANDLE]\n                handle=kernel.OpenProcess(0x1000,False,int(info[\'pid\']))\n                if handle:\n                    code=wintypes.DWORD()\n                    alive=bool(kernel.GetExitCodeProcess(handle,ctypes.byref(code))) and code.value==259\n                    kernel.CloseHandle(handle)\n            else:\n                try:os.kill(info[\'pid\'],0);alive=True\n                except OSError:pass\n        if alive:raise RuntimeError(f\'Output is in use by PID {info["pid"]}\')\n        lock.unlink()\n    descriptor=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)\n    os.write(descriptor,json.dumps(dict(pid=os.getpid(),host=socket.gethostname())).encode());os.close(descriptor)\n    try:yield\n    finally:lock.unlink(missing_ok=True)\n\n\ndef run_cv(data_dir=None, output_dir=\'task4_v2_results\', cfg=None, folds_to_run=(0,1,2,3,4),\n           weights_path=None, resume_from=None, backups=True, full_backup=True):\n    cfg=cfg or ConfigV2()\n    if cfg.folds!=5 or cfg.fold!=0 or cfg.patch<64 or cfg.patch%32 or cfg.patch>512:\n        raise ValueError(\'Require 5 folds, audit fold=0, patch multiple of 32 in [64,512].\')\n    if cfg.epochs<1 or cfg.batch_size<1 or cfg.patches_per_epoch<1 or cfg.monitor_every<1:\n        raise ValueError(\'Invalid training configuration\')\n    if not 0<cfg.ema_decay<1 or len(set(folds_to_run))!=len(folds_to_run) or not set(folds_to_run)<=set(range(5)):\n        raise ValueError(\'Invalid EMA decay/fold list\')\n    torch.set_num_threads(cfg.threads);cv2.setNumThreads(0)\n    output=Path(output_dir).resolve()\n    if resume_from:restore_output(resume_from,output)\n    output.mkdir(parents=True,exist_ok=True)\n    root=base.find_data(data_dir)\n    identity=dict(config=asdict(cfg),source_sha256=base.file_sha(__file__),\n                  baseline_helpers_sha256=base.file_sha(base.__file__))\n    config_path=output/\'run_config.json\'\n    previous=json.loads(config_path.read_text(encoding=\'utf-8\')) if config_path.exists() else None\n    if previous and any(previous.get(k)!=v for k,v in identity.items()):\n        raise ValueError(\'Changed config/source: use a NEW OUTPUT_DIR to preserve prior results.\')\n    with run_lock(output):\n        train,test,labels,assignment,ids,audit=audit_v2(root,output,cfg)\n        identity[\'data_fingerprint\']=audit[\'data_fingerprint\']\n        if previous and previous[\'data_fingerprint\']!=identity[\'data_fingerprint\']:\n            raise ValueError(\'Dataset changed: use a new output directory\')\n        initial_state,provenance=resolve_weights(cfg,output,weights_path)\n        identity[\'pretrained_sha256\']=provenance[\'sha256\']\n        if previous and previous!=identity:raise ValueError(\'Pretrained weights differ from saved run\')\n        base.save_json(config_path,identity)\n        base.save_json(output/\'pretrained_source.json\',provenance)\n        base.save_json(output/\'environment.json\',dict(python=platform.python_version(),torch=torch.__version__,\n            torchvision=torchvision.__version__,opencv=cv2.__version__,numpy=np.__version__,\n            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n            training_device_count=1,torch_cuda=torch.version.cuda))\n        code_dir=output/\'runtime_source\';code_dir.mkdir(exist_ok=True)\n        for p in [Path(__file__),Path(base.__file__)]:\n            if p.resolve()!=(code_dir/p.name).resolve():shutil.copyfile(p,code_dir/p.name)\n        device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        print(\'Device:\',device,\'fixed epochs:\',cfg.epochs,\'folds requested:\',folds_to_run,flush=True)\n        if device.type!=\'cuda\':print(\'CPU detected; formal five-fold training will be slow. Enable GPU.\',flush=True)\n        for fold in folds_to_run:\n            folder=output/f\'fold{fold}\'\n            if (folder/\'completed.json\').exists():\n                print(f\'fold{fold}: completed, reusing verified artifacts\',flush=True)\n                continue\n            development=[p for p in train if assignment[p.stem]!=fold]\n            validation=[p for p in train if assignment[p.stem]==fold]\n            model=train_fold(development,validation,labels,cfg,fold,folder,initial_state,device)\n            infer_fold(model,validation,test,labels,cfg,folder,device)\n            del model\n            if device.type==\'cuda\':torch.cuda.empty_cache()\n        summary=aggregate(root,output,cfg,train,test,labels,assignment,ids)\n        if backups:make_backups(output,full=full_backup)\n    return output,summary\n'
for filename, text in [('task4_pipeline.py', BASE_SOURCE), ('task4_kaggle_v2.py', V2_SOURCE)]:
    (RUNTIME_DIR / filename).write_text(text, encoding='utf-8', newline='\n')

def load_module(name, filename):
    spec = importlib.util.spec_from_file_location(name, RUNTIME_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

BASE = load_module('task4_pipeline', 'task4_pipeline.py')
V2 = load_module('task4_v2_impl', 'task4_kaggle_v2.py')
print('Embedded implementations loaded:', RUNTIME_DIR)


## 3. 数据与GPU预检查


In [ ]:
from dataclasses import replace
import json
import torch

CFG = V2.ConfigV2(seed=SEED, epochs=EPOCHS, patch=PATCH_SIZE,
                  patches_per_epoch=PATCHES_PER_EPOCH, batch_size=BATCH_SIZE,
                  pretrained=PRETRAINED)
RUN_FOLDS = tuple(FOLDS_TO_RUN)
if SMOKE_MODE:
    CFG = replace(CFG, epochs=1, patches_per_epoch=8, batch_size=min(2, BATCH_SIZE), monitor_every=1)
    RUN_FOLDS = (0,)
    OUTPUT_DIR = OUTPUT_DIR.with_name(OUTPUT_DIR.name + '_smoke')
    RESUME_FROM = None

ROOT = BASE.find_data(DATA_DIR)
AUDIT_DIR = RUNTIME_DIR / 'preflight_audit'  # keep restore target empty until RESUME_FROM is copied
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN, TEST, LABELS, SPLIT, TEMPLATE_IDS, AUDIT = V2.audit_v2(ROOT, AUDIT_DIR, CFG)
print('Data:', ROOT)
print('Config:', CFG)
print('Output:', OUTPUT_DIR)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Folds:', {k: [p.stem for p in TRAIN if SPLIT[p.stem] == k] for k in range(5)})
assert set(AUDIT['validation_ids']) == {'2','6','9','10'}, 'Dataset/split differs from the audited first baseline.'
if RUN_TRAINING and not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before formal training.')


In [ ]:
# 合成样本检查几何反变换与实际GPU梯度，不把这里的随机模型当正式成绩。
import numpy as np

marker = torch.arange(2*3*64*64).reshape(2,3,64,64)
for rotation in range(4):
    for flip in [False, True]:
        assert torch.equal(V2.inverse_d4(V2.transform_d4(marker, rotation, flip), rotation, flip), marker)

if RUN_TRAINING:
    DEVICE = torch.device('cuda')
    probe = V2.ResNet34UNet().to(DEVICE).train()
    optimizer = torch.optim.AdamW(probe.parameters(), lr=1e-4)
    before = probe.head.weight.detach().clone()
    image = torch.rand(2,3,64,64,device=DEVICE)
    target = torch.zeros(2,1,64,64,device=DEVICE)
    target[:,:,12:16,10:54] = 1
    loss = V2.objective(probe(image), target)
    loss.backward()
    assert torch.isfinite(loss)
    assert all(torch.isfinite(p.grad).all() for p in probe.parameters() if p.grad is not None)
    optimizer.step()
    assert not torch.equal(before, probe.head.weight)
    del probe, optimizer, image, target, loss, before
    torch.cuda.empty_cache()
print('D4 inverse and GPU forward/backward preflight passed.' if RUN_TRAINING else 'D4 inverse passed; training disabled.')


## 4. 五折训练、预测、校准与打包

每轮保存`last.pt`（模型、EMA、优化器、混合精度、随机数和进度）。每折完成后保存固定最终EMA、验证/测试概率和完成标记。训练中断后，同配置再次运行即可续接；已完成折直接复用并核验文件哈希。

Kaggle新会话要先把上次输出挂载为输入，再将`RESUME_FROM`设为完整结果目录或`full_backup.zip`；不要用轻量分析包续训。完整备份约数GB，默认生成并核验ZIP CRC。结果保存在`/kaggle/working`，使用Save Version保留，并从Output面板下载。

所有折完成才汇总20图OOF并导出主提交。默认首选文件是`submission.csv`；同时保留无TTA、D4两份对照，以及独立命名的转码代理候选。程序只生成文件，不自动提交比赛。


In [ ]:
if RUN_TRAINING:
    RESULT_DIR, SUMMARY = V2.run_cv(
        data_dir=ROOT, output_dir=OUTPUT_DIR, cfg=CFG, folds_to_run=RUN_FOLDS,
        weights_path=WEIGHTS_PATH, resume_from=RESUME_FROM,
        backups=True, full_backup=SAVE_FULL_BACKUP)
else:
    RESULT_DIR, SUMMARY = OUTPUT_DIR, None


## 5. 查看与下载


In [ ]:
from IPython.display import display, Image as DisplayImage, FileLink
import pandas as pd

if (RESULT_DIR / 'cv_summary.json').exists():
    SUMMARY = json.loads((RESULT_DIR / 'cv_summary.json').read_text(encoding='utf-8'))
    display(pd.DataFrame(SUMMARY['modes']).T)
    print('Selected mode:', SUMMARY['selected_mode'])
    print('Matched first-fold reference:', SUMMARY['baseline_reference'])
    for filename in ['oof_fold0_preview.png', 'test_preview.png']:
        display(DisplayImage(filename=str(RESULT_DIR / filename)))
    for filename in ['submission.csv', 'submission_none.csv', 'submission_d4.csv',
                     'submission_converter_proxy.csv', 'analysis_results.zip', 'full_backup.zip']:
        if (RESULT_DIR / filename).exists():
            display(FileLink(str(RESULT_DIR / filename)))
else:
    print('Full five-fold OOF is not ready. Partial/smoke outputs:', RESULT_DIR)


## 6. 跑完后保留什么

- `submission.csv`：基于OOF选择推理模式和干净血管阈值的主候选，20行，RLE与原官方转换器逐条相同。
- `cv_summary.json`与`oof_per_image.csv`：固定0.5、阈值校准、原尺寸指标及同图对照。关注TTA是否真的改善，并查看各折波动。
- `analysis_results.zip`：适合传回分析，包含分数、日志、CSV、预览和运行源码；不含模型权重与概率数组。
- `full_backup.zip`：包含全部模型、续训状态、概率数组及官方初始权重，另行保存，便于推理重放或恢复。

下一步应结合这轮完整OOF与首份公开分数，再决定增加训练量、换输入预处理或加入第二个模型。不要仅因全局调参后的OOF分数较高就认定比赛一定提升。
